# Домашнее задание 2


## 1

In [9]:
import numpy as np


def matrix_generate(m, n, *, kind='full', eps=0.0, rank=None, random_state=None): 
    """
    kind: 'full','diagonal','upper','lower','symmetric','singular','rank','perturb'
    eps: amplitude of uniform perturbation for 'perturb'.
    rank: integer rank for 'rank'
    """
    rng = np.random.default_rng(random_state)
    if kind == 'full':
        return rng.random((m, n))
    if kind == 'diagonal':
        if m != n:
            raise ValueError('diagonal requires square matrix')
        D = np.zeros((m, n))
        D[np.diag_indices(m)] = rng.random(m)
        return D
    if kind == 'upper':
        A = rng.random((m, n))
        return np.triu(A)
    if kind == 'lower':
        A = rng.random((m, n))
        return np.tril(A)
    if kind == 'symmetric':
        if m != n:
            raise ValueError('symmetric requires square matrix')
        A = rng.random((m, m))
        return 0.5 * (A + A.T)
    if kind == 'singular':
        A = rng.random((m, n))
        if m >= 2:
            A[1] = A[0]
        else:
            A[0] = 0
        return A
    if kind == 'rank':
        if rank is None:
            raise ValueError('rank must be provided for kind="rank"')
        U = rng.random((m, rank))
        V = rng.random((rank, n))
        return U @ V
    if kind == 'perturb':
        A = rng.random((m, n))
        P = rng.uniform(-eps, eps, size=(m, n))
        return A + P
    raise ValueError(f'Unknown kind: {kind}')

# quick demonstration
if __name__ == '__main__':
    print('full 3x4:\n', matrix_generate(3,4, kind='full', random_state=1))
    print('\nupper 4x3:\n', matrix_generate(4,3, kind='upper', random_state=1))
    print('\nsymmetric 4x4:\n', matrix_generate(4,4, kind='symmetric', random_state=2))
    print('\ndiagonal 4x4:\n', matrix_generate(4,4, kind='diagonal', random_state=3))
    print('\nrank 4x4 r=2, rank:', np.linalg.matrix_rank(matrix_generate(4,4, kind='rank', rank=2, random_state=4)))


full 3x4:
 [[0.51182162 0.9504637  0.14415961 0.94864945]
 [0.31183145 0.42332645 0.82770259 0.40919914]
 [0.54959369 0.02755911 0.75351311 0.53814331]]

upper 4x3:
 [[0.51182162 0.9504637  0.14415961]
 [0.         0.31183145 0.42332645]
 [0.         0.         0.54959369]
 [0.         0.         0.        ]]

symmetric 4x4:
 [[0.26161213 0.44929583 0.54459755 0.26227337]
 [0.44929583 0.72856053 0.42266704 0.36222196]
 [0.54459755 0.42266704 0.56226566 0.28642347]
 [0.26227337 0.36222196 0.28642347 0.6331844 ]]

diagonal 4x4:
 [[0.08564917 0.         0.         0.        ]
 [0.         0.23681051 0.         0.        ]
 [0.         0.         0.80127447 0.        ]
 [0.         0.         0.         0.58216204]]

rank 4x4 r=2, rank: 2


## 2

In [10]:
import numpy as np
from numpy.linalg import svd, inv


def vec_norm(x, p=2):
    x = np.asarray(x)
    if p == 1:
        return np.sum(np.abs(x))
    if p == 2:
        return np.sqrt(np.sum(np.abs(x)**2))
    if p == np.inf:
        return np.max(np.abs(x))
    raise ValueError('unsupported p')

def induced_mat_norm(A, p=2):
    A = np.asarray(A)
    if p == 1:
        return np.max(np.sum(np.abs(A), axis=0))
    if p == np.inf:
        return np.max(np.sum(np.abs(A), axis=1))
    if p == 2:
        s = svd(A, compute_uv=False)
        return s[0]
    raise ValueError('unsupported p')

def cond_number(A, p=2):
    A = np.asarray(A)
    try:
        A_inv = inv(A)
    except Exception:
        return np.inf
    return induced_mat_norm(A, p) * induced_mat_norm(A_inv, p)

# demonstration
if __name__ == '__main__':
    A = np.array([[1.,2.],[3.,4.]])
    print('vector norms for [1,2]:', vec_norm([1,2],1), vec_norm([1,2],2), vec_norm([1,2],np.inf))
    print('induced norms for A:', induced_mat_norm(A,1), induced_mat_norm(A,2), induced_mat_norm(A,np.inf))
    print('condition cond_2(A):', cond_number(A,2))


vector norms for [1,2]: 3 2.23606797749979 2
induced norms for A: 6.0 5.464985704219043 7.0
condition cond_2(A): 14.93303437365925


## 3

Для любого вектора $x\in\mathbb{R}^m$ справедливо
$$
||x||_2 \le ||x||_1 \le \sqrt{m}\,||x||_2.
$$
Следовательно можно взять константы $C_1=1$, $C_2=\sqrt{m}$.
Проверим это численно:

In [11]:
import numpy as np


def equivalence_constants(m):
    C1 = 1.0
    C2 = np.sqrt(m)
    return C1, C2

# numeric check
if __name__ == '__main__':
    for m in [1,2,4,10,100]:
        C1,C2 = equivalence_constants(m)
        print(f'm={m}: C1={C1}, C2={C2:.6f}')
    # random verification
    m = 10
    C1,C2 = equivalence_constants(m)
    for _ in range(5):
        x = np.random.randn(m)
        assert np.linalg.norm(x,2) <= C1 * np.linalg.norm(x,1) + 1e-12
        assert np.linalg.norm(x,1) <= C2 * np.linalg.norm(x,2) + 1e-12
    print('Random checks passed')


m=1: C1=1.0, C2=1.000000
m=2: C1=1.0, C2=1.414214
m=4: C1=1.0, C2=2.000000
m=10: C1=1.0, C2=3.162278
m=100: C1=1.0, C2=10.000000
Random checks passed


# 4

### Доказательство векторного неравенства

Для каждого $i$ выполняется $|x_i| \le \|x\|_\infty$, тогда
$$
\|x\|_2^2 = \sum_{i=1}^m x_i^2 \le \sum_{i=1}^m \|x\|_\infty^2 = m \|x\|_\infty^2.
$$
Следовательно,
$$
\|x\|_2 \le \sqrt{m}\,\|x\|_\infty.
$$

Равенство достигается на векторах вида $x=(c,c,\dots,c)$.

---

### Доказательство матричного неравенства

Пусть строки матрицы $A$ — это $r_1,\dots,r_m$. Тогда
$$
\|A\|_\infty = \max_i \|r_i\|_1 \le \sqrt{n}\,\max_i \|r_i\|_2.
$$

Из определения спектральной нормы:
$$
\|A\|_2 = \sup_{\|x\|_2=1} \|Ax\|_2,
$$
для $x = r_i / \|r_i\|_2$ получаем
$$
\|Ax\|_2 \ge |(Ax)_i| = \|r_i\|_2,
$$
поэтому
$$
\max_i \|r_i\|_2 \le \|A\|_2.
$$

Следовательно,
$$
\|A\|_\infty \le \sqrt{n}\,\|A\|_2.
$$

Равенство достигается если у матрицы только одна ненулевая строка, в которой все элементы равны по модулю.


# 5

Рассмотрим квадрат нормы произведения $(UA)$:  
$$
\|U A\|_F^2=\operatorname{trace}\big((U A)(U A)^H\big)
=\operatorname{trace}\big(U A A^H U^H\big).
$$ 
Свойство следа: $(\operatorname{trace}(X Y)=\operatorname{trace}(Y X)). Перенесём (U^H) и (U)$:  
$$
\operatorname{trace}\big(U A A^H U^H\big)=\operatorname{trace}\big(A A^H U^H U\big).
$$  
Так как $U^H U = I$, получаем  
$$
\operatorname{trace}\big(A A^H U^H U\big)=\operatorname{trace}(A A^H)=\|A\|_F^2.
$$  
Отсюда $(\|U A\|_F=\|A\|_F)$.

Аналогично для произведения справа:  
$$
\|A U\|_F^2=\operatorname{trace}\big((A U)(A U)^H\big)
=\operatorname{trace}\big(A U U^H A^H\big)
=\operatorname{trace}\big(A A^H\big)=\|A\|_F^2,
$$  
поскольку $(U U^H=I)$. Значит $(\|A U\|_F=\|A\|_F)$.

Проверим численно:

In [12]:
import numpy as np
from numpy.linalg import norm

if __name__ == '__main__':
    rng = np.random.default_rng(0)
    A = rng.standard_normal((5,3))
    # orthogonal Q via QR
    Q, _ = np.linalg.qr(rng.standard_normal((5,5)))
    print('||Q A||_F =', norm(Q@A,'fro'), '||A||_F =', norm(A,'fro'))
    # right multiply by orthogonal of size 3x3
    Q3, _ = np.linalg.qr(rng.standard_normal((3,3)))
    print('||A Q||_F =', norm(A@Q3,'fro'), '||A||_F =', norm(A,'fro'))


||Q A||_F = 3.600294679065858 ||A||_F = 3.600294679065857
||A Q||_F = 3.6002946790658577 ||A||_F = 3.600294679065857
